In [1]:
import numpy as np
from ultralytics import YOLO
import easyocr
import pyttsx3
import threading
import queue
import time
import logging

# Configure logging
t = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format='[%(asctime)s]
%(levelname)s: %(message)s')

# Parameters
CONF_THRESHOLD = 0.5
FRAME_SIZE = (640, 640)
PADDING = 10 # pixels around detected box
DEVICE_INDEX = 0
YOLO_WEIGHTS = "yolov8n.pt"

# Thread-safe queue for OCR tasks
ocr_queue = queue.Queue()
stop_event = threading.Event()

# Initialize TTS engine
tts = pyttsx3.init()
tts.setProperty('rate', 150)
tts.setProperty('volume', 1.0)

# Initialize OCR reader (on-device fallback)
reader = easyocr.Reader(['en'], gpu=False)

# Initialize YOLOv8 model
t = logging.getLogger('ultralytics')
t.setLevel(logging.WARNING)
model = YOLO(YOLO_WEIGHTS)

# Speak function runs in separate thread
def speak(text: str):
 tts.say(text)
 tts.runAndWait()

# Worker thread for OCR and TTS
def ocr_worker():
    while not stop_event.is_set():
        try:
            region, label = ocr_queue.get(timeout=0.1)
        except queue.Empty:
            continue
            text_result = reader.readtext(region, detail=0)
            output = label
            n if text_result:
            output += ": " + " ".join(text_result)
            logging.info(f"OCR result: {output}")
            speak(output)
            ocr_queue.task_done()

# Start OCR worker thread
threading.Thread(target=ocr_worker, daemon=True).start()

# Main video loop
cap = cv2.VideoCapture(DEVICE_INDEX)
if not cap.isOpened():
    raise RuntimeError(f"Failed to open camera index {DEVICE_INDEX}"
                       try:
 while True:
     ret, frame = cap.read()
     if not ret:
         logging.warning("Frame capture failed, stopping.")
         break
         # Resize frame for inference
         resized = cv2.resize(frame, FRAME_SIZE)
         # YOLO detection
         results = model(resized)[0]
         for box in results.boxes:
             conf = float(box.conf[0])
             if conf < CONF_THRESHOLD:
                 continue
                 x1, y1, x2, y2 = map(int, box.xyxy[0])
                 # Add padding and clamp
                 x1 = max(0, x1-PADDING)
                 y1 = max(0, y1-PADDING)
                 x2 = min(FRAME_SIZE[0], x2+PADDING)
                 y2 = min(FRAME_SIZE[1], y2+PADDING)
                 # Crop ROI for OCR
                 roi = cv2.cvtColor(resized[y1:y2, x1:x2], cv2.COLOR_BGR2GRAY)
                 \roi = cv2.equalizeHist(roi)
                 _, roi = cv2.threshold(roi, 0, 255, cv2.THRESH_BINARY 
                                        cv2.THRESH_OTSU)
                 label = model.names[int(box.cls[0])]
                 # Enqueue for OCR processing
                 ocr_queue.put((roi, label))
                 # Draw bounding box and label
                 cv2.rectangle(resized, (x1, y1), (x2, y2), (0, 255, 0), 2)
                 cv2.putText(resized, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX,
                             0.5, (0, 255, 0), 2)
                 cv2.imshow("Assistive Vision", resized)
                 if cv2.waitKey(1) & 0xFF == 27: # ESC key
                     break
finally:
stop_event.set()
ocr_queue.join()
cap.release()
cv2.destroyAllWindows()